#use llamafactory environment

In [3]:
from datasets import load_dataset
import pandas as pd
import json


/n/home07/than157/.conda/envs/llamafactory/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#load training set of hellaswag
dataset = load_dataset("baber/piqa", split="train")

#print dataset info
print("Dataset info:")
print(dataset)

print("# samples:", len(dataset)) #should be 39k
n_samples = len(dataset)

#convert to dataframe
df = dataset.to_pandas()
df.head()


Dataset info:
Dataset({
    features: ['goal', 'sol1', 'sol2', 'label'],
    num_rows: 16113
})
# samples: 16113


,goal,sol1,sol2,label
0,"When boiling butter, when it's ready, you can",Pour it onto a plate,Pour it into a jar,1
1,"To permanently attach metal legs to a chair, y...",Weld the metal together to get it to stay firm...,Nail the metal together to get it to stay firm...,0
2,how do you indent something?,leave a space before starting the writing,press the spacebar,0
3,how do you shake something?,move it up and down and side to side quickly.,stir it very quickly.,0
4,Clean tires,"Pour water, cape off caked on dirt. Use speed...","Pour water, scrape off caked on dirt. Use a st...",1


In [5]:
#add 'answer' column: for each row, take either 'sol1' or 'sol2' column as the answer based on the 'label' column
df['answer'] = df.apply(lambda row: row['sol1'] if row['label'] == 0 else row['sol2'], axis=1)
df.head()

,goal,sol1,sol2,label,answer
0,"When boiling butter, when it's ready, you can",Pour it onto a plate,Pour it into a jar,1,Pour it into a jar
1,"To permanently attach metal legs to a chair, y...",Weld the metal together to get it to stay firm...,Nail the metal together to get it to stay firm...,0,Weld the metal together to get it to stay firm...
2,how do you indent something?,leave a space before starting the writing,press the spacebar,0,leave a space before starting the writing
3,how do you shake something?,move it up and down and side to side quickly.,stir it very quickly.,0,move it up and down and side to side quickly.
4,Clean tires,"Pour water, cape off caked on dirt. Use speed...","Pour water, scrape off caked on dirt. Use a st...",1,"Pour water, scrape off caked on dirt. Use a st..."


In [6]:
### create json file

#format data for sft
data = []

for idx, row in df.iterrows():
    item = {
        "instruction": row["goal"],
        "input": "",
        "output": row["answer"]
    }
    data.append(item)
    #track progress
    if idx % 10000 == 0:
        print(f"Processed {idx} rows")
    if idx == (n_samples - 1):
        print(f"Processed {idx} rows (last row)")

#save to JSON file
with open("data/piqa.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("Complete!")

Processed 0 rows
Processed 10000 rows
Processed 16112 rows (last row)
Complete!


### check that train split does not overlap with test questions in lm-eval-harness
yes because splits are specified in evolm/evaluation/lm-evaluation-harness/lm_eval/tasks/piqa/piqa.yaml and evolm/evaluation/lm-evaluation-harness/jobs/eval-single--run-exp.sh uses val split